In [0]:
WITH x AS (
  SELECT DateTime, date_trunc('MONTH', DateTime) AS month,
         CASE WHEN DateTime < '2026-03-17' THEN PV1_MW ELSE PV2_MW END AS act,
         Pv2_AVL AS avl, PV2_GHI AS ghi
  FROM ewec_dev_powerops.bronze.power_contango_minute_pv
  WHERE hour(DateTime) BETWEEN 8 AND 15 AND abs(PV1_MW) < 5000 AND abs(PV2_MW) < 5000
),
d AS (
  SELECT *, act - lag(act) OVER (ORDER BY DateTime) AS d_act,
            avl - lag(avl) OVER (ORDER BY DateTime) AS d_avl
  FROM x
)
SELECT month,
       ROUND(100 * AVG(CASE WHEN avl IS NULL THEN 1 ELSE 0 END), 1) AS avl_blank_pct,
       ROUND(100 * AVG(CASE WHEN act IS NULL THEN 1 ELSE 0 END), 1) AS act_blank_pct,
       ROUND(percentile_approx(act, 0.5)) AS act_median, ROUND(percentile_approx(avl, 0.5)) AS avl_median,
       ROUND(stddev(d_act), 1) AS act_jump_std, ROUND(stddev(d_avl), 1) AS avl_jump_std,
       ROUND(corr(act, ghi), 3) AS act_corr_sunlight, ROUND(corr(avl, ghi), 3) AS avl_corr_sunlight,
       ROUND(AVG(avl - act), 1) AS avg_avl_minus_act,
       ROUND(100 * AVG(CASE WHEN avl IN (1500, 1640) THEN 1 ELSE 0 END), 1) AS avl_at_cap_pct
FROM d GROUP BY month ORDER BY month;